# Data Cleaning — Craigslist Used Cars Dataset
**CT118-3-3 ODL | Assignment 1 — Task 2: Data Preparation**

Dataset: [Craigslist Used Cars & Trucks (Kaggle)](https://www.kaggle.com/datasets/austinreese/craigslist-carstrucks-data) —
426,880 listings × 26 columns. Input `dataset/vehicles.csv`, output `dataset/vehicles_cleaned.csv`,
which is consumed by `model_building_tuning_evaluation.ipynb`.

This notebook turns the noise findings from `eda.ipynb` into a reproducible cleaning
pipeline. Every decision below is stated with its justification and the number of rows it
costs, and the running row count is collected into a **cleaning log** (Section 11) so the
whole pipeline can be reported as a single table.

---

## What this notebook fixes relative to the first cleaning pass

| # | Issue found in the first pass | Action taken here | Section |
|---|---|---|---|
| 1 | **Duplicate listings.** The EDA reported *0 exact duplicates*, but only because `id`, `url` and `image_url` are unique per row. Dealers repost the same physical car across Craigslist regions — VIN `1FMJU1JT1HEA52352` appears 261 times. Because `VIN` was dropped *before* the duplicate check, every repost survived as its own row. The same car then lands in both the train and test split, so test metrics flatter the model (**leakage**, not just redundancy). | Deduplicate on `VIN` **while the column still exists**; the ~38% of rows with no VIN are deduplicated separately on a composite key. | 4 |
| 2 | **Unrealistically low prices.** `price > 0` removed the 32,895 zero-price rows but kept `$1`, `$123`, `$500` dealer bait listings. With MAPE as a reported metric, a `$1` listing predicted at `$8,000` contributes an absolute percentage error of 799,900% — a handful of these dominate the metric. | Floor the price at **$500** (the floor the EDA next-steps section already recommended) and keep the IQR upper fence of $57,364. | 3.1 |
| 3 | **`description` survived into the cleaned CSV.** It is picked up by `select_dtypes(include=['object'])` and treated as a categorical column with 360,911 unique values — it would blow up or produce a nonsense embedding table. It is also a leakage risk, because sellers routinely write the price into the description text. | Drop `description`. Drop `posting_date` too, after using it to derive the scrape year (Section 5). | 5 |
| 4 | **Filters silently dropped rows meant for imputation.** `NaN >= 1995` and `NaN <= 277300` both evaluate to `False`, so every row with a missing `year` or `odometer` was dropped by the outlier filter — the later median fill for those two columns never fired. The notebook said one thing and the code did another. | Handle the NaNs **explicitly and deliberately** before the range filters, and log how many rows that costs. | 3.2 |
| 5 | **`odometer == 0`.** 1,965 rows. A used car with zero miles is a data-entry error or means "not stated"; leaving it in teaches the network that zero mileage is normal and cheap. | Convert `0` to `NaN` and handle it under one explicit policy with the other missing odometers. | 3.2 |
| 6 | **`model` cardinality is unusable.** 29,667 unique values, with `f-150`, `f150`, `F-150 XLT` and `f 150 supercrew` all denoting the same vehicle. Fed straight to an embedding layer that is a 29,667-row table whose rows are mostly seen once or twice — memorised noise and an inflated parameter count. | Normalise the text, then collapse the long tail to the top **N** models (N chosen from a printed coverage table) with everything else as `other`. | 6 |
| 7 | **`cylinders` is text, not a number.** Values look like `"6 cylinders"`. There is a real ordinal relationship (4 < 6 < 8) that is thrown away by treating it categorically. | Extract the integer into `cylinders_num`, and keep `"other"`/missing as its own binary flag. | 7 |
| 8 | **`size` is 72% missing** and was being filled with `'unknown'`, leaving a column that is three-quarters one value. | Dropped, consistent with the EDA recommendation (toggle in the config). | 8 |
| 9 | **`lat`/`long` median imputation** places every missing-coordinate car at a single point in Missouri — an invented location. `region` and `state` already encode geography and will be embedded. | Dropped (toggle in the config). | 8 |
| 10 | **Feature engineering outstanding.** Two items from the EDA next-steps list. | Add `car_age` (from the **scrape year**, not the current year) and `price_log` = `log1p(price)`. | 9 |

**Deliberately left out of this notebook:** scaling and label/ordinal encoding. Fitting a
`StandardScaler` or an encoder on the full dataset here would leak validation and test
statistics into training. The modelling notebook splits first and fits those on the training
split only — see Section 12.


## 1. Imports & Configuration

Every threshold is a named constant so the cleaning policy can be read (and tuned) in one
place instead of being buried in the code. Paths use `os.path.join`, so the notebook runs on
Windows, macOS and Linux alike.


In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 40)

# ── PATHS ─────────────────────────────────────────────────────────────────────
DATA_PATH       = os.path.join('dataset', 'vehicles.csv')
CLEAN_DATA_PATH = os.path.join('dataset', 'vehicles_cleaned.csv')

# ── CLEANING POLICY ───────────────────────────────────────────────────────────
# Price: lower bound rejects dealer bait listings ($1 / $123 placeholder prices);
# upper bound is the IQR upper fence computed in eda.ipynb.
PRICE_MIN, PRICE_MAX = 500, 57_364

# Year / odometer: IQR fences from eda.ipynb (year fence 1994.5, odometer fence 277,300.2).
YEAR_MIN     = 1995
ODOMETER_MAX = 277_300

# How to treat missing year / odometer, and odometer == 0 ('not stated' or a typo).
#   'drop'   -> remove the rows (they are ~1.5% of the data and cannot be recovered)
#   'impute' -> median fill, keeping the rows
MISSING_NUMERIC_POLICY = 'drop'

# Long-tail collapse for `model`; see the coverage table in Section 6 before changing.
TOP_N_MODELS = 500

# Columns the EDA flagged but the first cleaning pass kept (see the table above).
DROP_SIZE     = True   # 72% missing
DROP_LATLONG  = True   # region/state already encode geography; median fill invents locations

# Fallback if `posting_date` cannot be parsed; the scrape year is normally read from the data.
FALLBACK_SCRAPE_YEAR = 2022

# Composite key used to deduplicate the rows that have no VIN.
COMPOSITE_KEY = ['manufacturer', 'model', 'year', 'odometer', 'price', 'region']

# Rows missing any of these cannot be modelled and are too few to impute meaningfully.
CRITICAL_COLS = ['manufacturer', 'model', 'fuel', 'transmission', 'title_status']


## 2. Load Raw Data & Set Up the Cleaning Log

`log_step()` records the row count after every stage. The resulting table (Section 11) is the
audit trail for the report: it shows exactly how many rows each decision cost, which is the
part of a cleaning pipeline that is normally invisible.


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
N_RAW = len(df)
print(f'Raw dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')

_log = []

def log_step(step, frame, note=''):
    '''Record rows remaining after a cleaning stage and print a one-line summary.'''
    prev    = _log[-1]['rows_after'] if _log else N_RAW
    removed = prev - len(frame)
    _log.append({
        'step'         : step,
        'rows_after'   : len(frame),
        'rows_removed' : removed,
        'pct_of_raw'   : round(100 * len(frame) / N_RAW, 2),
        'note'         : note,
    })
    print(f'{step:<46s} rows: {len(frame):>7,}   (removed {removed:>6,})')

log_step('0. Raw data loaded', df)


## 3. Row Validity Filters

These run **before** deduplication so that the representative kept for each duplicate group is
a valid listing rather than an arbitrary one (a `$0` repost, say).


### 3.1 Price — floor at \$500, cap at the IQR upper fence

The old filter was `price > 0`. That cleared the 32,895 zero-price rows but left the `$1` and
`$123` bait listings intact. Those rows are not cheap cars; they are missing prices encoded as
a number, and because the target is the price they are unlearnable noise on the label itself.

The MAPE case is the decisive one. MAPE divides the absolute error by the true value, so a
`$1` listing predicted at a perfectly reasonable `$8,000` contributes
`|8000 − 1| / 1 = 799,900%`. A few dozen such rows swamp the mean and the reported MAPE stops
describing the model at all.


In [ ]:
n_zero  = (df['price'] == 0).sum()
n_bait  = ((df['price'] > 0) & (df['price'] < PRICE_MIN)).sum()
n_upper = (df['price'] > PRICE_MAX).sum()
print(f'{"price == 0":<34s}: {n_zero:>7,}')
print(f'{f"0 < price < ${PRICE_MIN:,}":<34s}: {n_bait:>7,}   <- kept by the old `price > 0` filter')
print(f'{f"price > ${PRICE_MAX:,} (IQR upper fence)":<34s}: {n_upper:>7,}')

df = df[(df['price'] >= PRICE_MIN) & (df['price'] <= PRICE_MAX)]
log_step(f'1. Price in [${PRICE_MIN:,}, ${PRICE_MAX:,}]', df)


### 3.2 Year and odometer — NaNs handled explicitly, then the range filters

The bug in the first pass was quiet: `NaN >= 1995` and `NaN <= 277300` are both `False`, so
`df[df['year'] >= 1995]` dropped every row with a missing `year`, and the median fill later in
the notebook never saw them. The effect was defensible (about 5,600 rows), but it happened by
accident, and the notebook's prose claimed those columns were imputed.

Here the NaNs are dealt with first, under an explicit policy, and counted. `odometer == 0` is
folded into the same decision: 1,965 used cars claim zero miles, which is either a typo or a
seller writing "not stated", so a zero is treated as a missing value rather than a real
reading. Leaving them in would teach the network that zero mileage is ordinary — and cheap.


In [ ]:
n_year_na = df['year'].isna().sum()
n_odo_na  = df['odometer'].isna().sum()
n_odo_0   = (df['odometer'] == 0).sum()
print(f'missing year            : {n_year_na:,}')
print(f'missing odometer        : {n_odo_na:,}')
print(f'odometer == 0           : {n_odo_0:,}   <- treated as missing, not as a real reading')

# A zero odometer reading is not information; make it explicit before the policy is applied.
df['odometer'] = df['odometer'].replace(0, np.nan)

if MISSING_NUMERIC_POLICY == 'drop':
    df = df.dropna(subset=['year', 'odometer'])
    note = 'missing year/odometer dropped (unrecoverable)'
else:
    df['year']     = df['year'].fillna(df['year'].median())
    df['odometer'] = df['odometer'].fillna(df['odometer'].median())
    note = 'missing year/odometer median-imputed'

log_step('2. Missing year/odometer resolved', df, note)

# Range filters now act only on real values, so nothing is dropped as a side effect.
df = df[(df['year'] >= YEAR_MIN) & (df['odometer'] <= ODOMETER_MAX)]
df['year'] = df['year'].astype(int)
log_step(f'3. year >= {YEAR_MIN}, odometer <= {ODOMETER_MAX:,}', df)


### 3.3 Rows missing a critical categorical

`manufacturer`, `model`, `fuel`, `transmission` and `title_status` carry the bulk of the
non-numeric signal, and an `'unknown'` category for them would be a large, meaningless
embedding bucket. These are dropped before deduplication so that the composite key in
Section 4 never has to compare `NaN` against `NaN` (pandas treats two `NaN`s as equal inside
`drop_duplicates`, which would merge genuinely different cars).


In [ ]:
print(df[CRITICAL_COLS].isna().sum().to_string())
df = df.dropna(subset=CRITICAL_COLS)
log_step('4. Rows missing a critical categorical', df)


## 4. Deduplication — the leakage fix

This is the most consequential change in the notebook.

The EDA reported **0 exact duplicate rows**, which is true and misleading at the same time.
`id`, `url` and `image_url` are generated per posting, so any two rows differ in at least three
columns even when they describe the same physical car. Dealers repost the same vehicle across
several Craigslist regions to widen their reach: VIN `1FMJU1JT1HEA52352` appears **261 times**.
The first pass dropped `VIN` in its very first step, so the duplicate check ran on a frame that
no longer contained the one column capable of revealing the problem.

The cost is not wasted rows, it is **leakage**. `train_test_split` shuffles rows, not vehicles,
so copies of one car land on both sides of the split. The network sees the car and its price
during training and is then scored on the same car at test time. Test MAE and R² come out
better than the model actually is, and the error is invisible — nothing in the metrics looks
wrong.

**Two-part strategy.** About 38% of rows have no VIN, and throwing those away would be a large,
avoidable loss. So:

* rows **with** a VIN are deduplicated on the normalised VIN;
* rows **without** a VIN are deduplicated on the composite key
  `['manufacturer', 'model', 'year', 'odometer', 'price', 'region']` — a car of the same make,
  model, year, mileage and price in the same region is the same listing for practical purposes.

The two halves are recombined and the original row order restored.


In [ ]:
# Normalise before matching: whitespace and case differences would hide real duplicates.
df['VIN'] = df['VIN'].astype('string').str.strip().str.upper().replace('', pd.NA)

has_vin = df['VIN'].notna()
print(f'rows with a VIN   : {has_vin.sum():,}  ({has_vin.mean()*100:.1f}%)')
print(f'rows without a VIN: {(~has_vin).sum():,}  ({(~has_vin).mean()*100:.1f}%)')

vin_counts = df.loc[has_vin, 'VIN'].value_counts()
print(f'\nDistinct VINs                     : {len(vin_counts):,}')
print(f'VINs appearing more than once     : {(vin_counts > 1).sum():,}')
print(f'Most-reposted VIN                 : {vin_counts.index[0]} x {vin_counts.iloc[0]:,}')
print('\nTop 5 most-reposted vehicles:')
print(vin_counts.head().to_string())


In [ ]:
df_vin   = df[has_vin].drop_duplicates(subset=['VIN'], keep='first')
df_novin = df[~has_vin].drop_duplicates(subset=COMPOSITE_KEY, keep='first')

print(f'VIN rows      : {has_vin.sum():>7,} -> {len(df_vin):>7,}  '
      f'({has_vin.sum() - len(df_vin):,} reposts removed)')
print(f'non-VIN rows  : {(~has_vin).sum():>7,} -> {len(df_novin):>7,}  '
      f'({(~has_vin).sum() - len(df_novin):,} duplicates removed)')

df = pd.concat([df_vin, df_novin]).sort_index()
log_step('5. Deduplicated (VIN + composite key)', df, 'removes train/test leakage')


## 5. Derive the Scrape Year, then Drop Text and Identifier Columns

`posting_date` is used once — to establish the reference year for `car_age` — and then dropped.
Reading the year from the data is safer than hard-coding it: the listings span only a few
months, and using the *current* year (2025) instead of the **scrape year** would add a constant
offset of several years to every age, distorting the depreciation curve the model is trying to
learn. `FALLBACK_SCRAPE_YEAR` covers the case where the column is unparseable.

**Dropped, with reasons:**

| Column | Reason |
|---|---|
| `description` | 360,911 unique values. `select_dtypes(include=['object'])` would sweep it into the categorical set and it would become a 360k-row embedding table. It is also a leakage risk — sellers frequently write the price into the description text. Out of scope unless the project adds an NLP branch. |
| `posting_date` | Unparsed string spanning a few months; the useful part is distilled into `car_age`. |
| `VIN` | Its job (deduplication, Section 4) is done; as a feature it is a unique identifier. |
| `id`, `url`, `region_url`, `image_url` | Identifiers and URLs; unique per row, zero predictive content. |
| `county` | 100% missing. |
| `size` | 72% missing (issue 8). Filling it with `'unknown'` leaves a column that is three-quarters one value — it will not help, and it costs an embedding table. |
| `lat`, `long` | 1.5% missing (issue 9). Median imputation would place every affected car at a single point in Missouri, which is invented data. `region` and `state` already encode geography and are embedded. |


In [ ]:
posting_year = pd.to_datetime(df['posting_date'], errors='coerce', utc=True).dt.year
if posting_year.notna().any():
    SCRAPE_YEAR = int(posting_year.max())
    print(f'Scrape year read from posting_date : {SCRAPE_YEAR} '
          f'(range {int(posting_year.min())}-{int(posting_year.max())})')
else:
    SCRAPE_YEAR = FALLBACK_SCRAPE_YEAR
    print(f'posting_date unparseable; falling back to {SCRAPE_YEAR}')

cols_to_drop = ['description', 'posting_date', 'VIN',
                'id', 'url', 'region_url', 'image_url', 'county']
if DROP_SIZE:
    cols_to_drop.append('size')
if DROP_LATLONG:
    cols_to_drop += ['lat', 'long']

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f'\nDropped {len(cols_to_drop)} columns -> {df.shape[1]} remain: {list(df.columns)}')
log_step('6. Dropped text / identifier columns', df, f'{df.shape[1]} columns remain')


## 6. `model` — Normalise the Text, Collapse the Long Tail

29,667 raw values, and the tail is mostly noise from free-text entry: `f-150`, `f150`,
`F-150 XLT` and `f 150 supercrew` are all the same truck. An embedding layer over that
vocabulary would be a 29,667 × d table whose rows are seen once or twice each — parameters
spent on memorising individual listings.

Two steps: lowercase, strip punctuation and collapse whitespace (which merges the spelling
variants), then keep the top `TOP_N_MODELS` and map the rest to `other`.

The coverage table below is the basis for choosing N rather than guessing it — it reports what
share of listings and of distinct values each cut-off retains.


In [ ]:
df['model'] = (df['model'].astype('string')
                          .str.lower()
                          .str.strip()
                          .str.replace(r'[^a-z0-9 ]', '', regex=True)   # punctuation -> gone
                          .str.replace(r'\s+', ' ', regex=True)         # f 150  -> f 150
                          .str.strip())
df['model'] = df['model'].replace('', pd.NA).fillna('unknown')

counts = df['model'].value_counts()
print(f'Distinct models after text normalisation: {counts.size:,} '
      f'(raw dataset: 29,667)\n')

print(f'{"top N":>8} {"coverage of rows":>18} {"rows -> other":>15}')
for n in [100, 250, 500, 1_000, 2_000, 5_000]:
    cov = counts.nlargest(n).sum() / len(df)
    print(f'{n:>8,} {cov*100:>17.1f}% {int((1-cov)*len(df)):>15,}')


In [ ]:
top_models  = counts.nlargest(TOP_N_MODELS).index
df['model'] = df['model'].where(df['model'].isin(top_models), 'other')

kept = (df['model'] != 'other').mean()
print(f"TOP_N_MODELS = {TOP_N_MODELS} -> {df['model'].nunique():,} categories "
      f"(incl. 'other'), covering {kept*100:.1f}% of rows directly")
print(f"\nMost common models:\n{df['model'].value_counts().head(8).to_string()}")
log_step(f'7. model normalised, top-{TOP_N_MODELS} kept', df,
         f"{df['model'].nunique():,} categories (was 29,667)")


## 7. `cylinders` — Recover the Ordinal Number

The raw values are strings such as `"6 cylinders"`, plus an `"other"` bucket. Treated as a
category, `4 cylinders`, `6 cylinders` and `8 cylinders` are three unrelated labels and the
network has to rediscover their ordering from data. The count is a genuine ordinal quantity
(and roughly proportional to engine size, so it is close to linear in price), so it belongs in
the numeric block.

`"other"` and missing values are not numbers and must not be silently median-filled into
looking like ordinary engines. They get their own binary flag, `cylinders_unknown`, so the
network can learn a separate offset for "we do not know" instead of being told these cars have
a 6-cylinder engine.


In [ ]:
raw_cyl = df['cylinders'].astype('string').str.lower()

df['cylinders_num']     = raw_cyl.str.extract(r'(\d+)', expand=False).astype(float)
df['cylinders_unknown'] = df['cylinders_num'].isna().astype(int)

n_other = raw_cyl.str.contains('other', na=False).sum()
print(f"'other' values      : {n_other:,}")
print(f'missing / unusable  : {df["cylinders_unknown"].sum():,} '
      f'({df["cylinders_unknown"].mean()*100:.1f}%) -> flagged, then median-filled')
print(f'\nExtracted distribution:\n{df["cylinders_num"].value_counts().sort_index().to_string()}')

# Fill the flagged rows so the column is numeric-complete; the flag preserves the distinction.
df['cylinders_num'] = df['cylinders_num'].fillna(df['cylinders_num'].median())
df = df.drop(columns=['cylinders'])
print(f'\nReplaced text `cylinders` with `cylinders_num` + `cylinders_unknown`')


## 8. Remaining Missing Values

After Sections 3, 5 and 7 the only columns still carrying gaps are low-signal categoricals —
`condition`, `drive`, `type`, `paint_color`. For these, missingness is itself informative (a
seller who omits the condition is telling you something), so `'unknown'` is kept as a real
category rather than imputed away with the mode.

Nothing numeric needs a median fill at this point: `year` and `odometer` were resolved under
an explicit policy in Section 3.2, `cylinders_num` in Section 7, and `lat`/`long` were dropped.
That is the difference from the first pass, where a blanket `fillna(median())` at the end
appeared to cover the numeric columns but in fact never fired for the two that mattered.


In [ ]:
missing_before = df.isna().sum()
print('Columns still containing missing values:')
print(missing_before[missing_before > 0].to_string() or '  (none)')

# exclude=['number'] selects the text columns under both pandas 2 (object) and 3 (str).
cat_cols = df.select_dtypes(exclude=['number']).columns
df[cat_cols] = df[cat_cols].fillna('unknown').astype(str)

assert df.isna().sum().sum() == 0, 'unexpected missing values remain'
log_step('8. Categorical gaps -> "unknown"', df, 'no missing values remain')


## 9. Feature Engineering

Two items carried over from the EDA next-steps list.

**`car_age = SCRAPE_YEAR − year`.** Depreciation is a function of how old a car was *when it
was listed*, not of its model year in the abstract. Age is also stable over time in a way the
raw year is not: a 2015 car means something different in a 2021 snapshot than in a 2025 one.
The reference year comes from `posting_date` (Section 5) rather than from `2025` — using the
current year would inflate every age by the same constant and misplace the whole curve. Ages
are clipped at 0, since next-model-year cars legitimately appear in the listings.

**`price_log = log1p(price)`.** Price stays right-skewed even after the $57k cap. Squared-error
loss on the raw target is dominated by the expensive tail, so the network spends its capacity
on the cars it will predict worst. Training on `log1p` stabilises the variance and generally
gives a clear improvement on this dataset. Both columns are written out: `price_log` is the
training target, and **raw `price` is kept so every metric can be computed in dollar space** —
`expm1` the predictions before scoring, otherwise MAE, RMSE, MAPE and R² are in log units and
comparable to nothing in the published literature.


In [ ]:
df['car_age']   = (SCRAPE_YEAR - df['year']).clip(lower=0)
df['price_log'] = np.log1p(df['price'])

print(f'car_age  (reference year {SCRAPE_YEAR}): '
      f'min {df["car_age"].min()}, median {df["car_age"].median():.0f}, max {df["car_age"].max()}')
print(f'price      skew: {df["price"].skew():>6.2f}')
print(f'log1p(price) skew: {df["price_log"].skew():>6.2f}   <- target used for training')


## 10. Final Validation

Cheap assertions that fail loudly here rather than silently corrupting the modelling notebook:
no missing values, no duplicate rows on the feature set, every range respected, and no
categorical column with a cardinality high enough to break an embedding layer.


In [ ]:
assert df.isna().sum().sum() == 0
assert df['price'].between(PRICE_MIN, PRICE_MAX).all()
assert df['year'].ge(YEAR_MIN).all()
assert df['odometer'].between(1, ODOMETER_MAX).all()
assert 'description' not in df.columns and 'VIN' not in df.columns

cat_cols = df.select_dtypes(exclude=['number']).columns
card = df[cat_cols].nunique().sort_values(ascending=False)
print('Categorical cardinality (embedding table size per column):')
print(card.to_string())
assert card.max() <= TOP_N_MODELS + 1, 'a categorical column is too high-cardinality to embed'

print(f'\nFinal shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.describe().T


## 11. Cleaning Log

The audit trail for the report — every stage, what it removed, and what fraction of the raw
426,880 listings survives at that point.


In [ ]:
log_df = pd.DataFrame(_log)
print(f'Raw rows      : {N_RAW:,}')
print(f'Cleaned rows  : {len(df):,}  ({len(df)/N_RAW*100:.1f}% retained)')
print(f'Rows removed  : {N_RAW - len(df):,}\n')
log_df


## 12. Save the Cleaned Dataset

Written to `dataset/vehicles_cleaned.csv` for `model_building_tuning_evaluation.ipynb`.

> **Deliberately not done here: scaling and encoding.** `StandardScaler`, `OrdinalEncoder` and
> `LabelEncoder` are all *fitted* objects — they learn statistics (means, variances, category
> vocabularies) from whatever data they are shown. Fitting them on the full dataset in this
> notebook would push validation and test statistics into the training inputs, which is the
> same class of leakage as the duplicate listings in Section 4, just harder to see. The
> modelling notebook performs the 70/15/15 split **first**, then fits the scaler and encoder on
> the training split only and applies them to the other two.


In [ ]:
os.makedirs(os.path.dirname(CLEAN_DATA_PATH) or '.', exist_ok=True)
df.to_csv(CLEAN_DATA_PATH, index=False)

size_mb = os.path.getsize(CLEAN_DATA_PATH) / 1e6
print(f'Saved {len(df):,} rows x {df.shape[1]} columns to {CLEAN_DATA_PATH} ({size_mb:,.1f} MB)')
df.head()
